# 05 — Modèles statistiques & séries temporelles

> **Question :** Les modèles économétriques apportent-ils quelque chose que les modèles ML ne peuvent pas voir ?

| | |
|---|---|
| **Hypothèse** | ARMA ne prédit pas bien le retour moyen, mais GARCH peut modéliser la volatilité (= l'incertitude) |
| **Données** | Série prix CBOT 2000-2025 |
| **Intérêt agricole** | Un modèle de volatilité permet d'adapter les seuils de décision au régime de risque |

## Familles testées

1. **AR(p)** — autocorrélation des returns
2. **ARIMA(p,d,q)** — avec intégration et MA
3. **SARIMAX** — ARIMA + saisonnalité + variables exogènes
4. **GARCH(1,1)** — modèle de volatilité conditionnelle
5. **Markov 2-états** — régimes latents (bull/non-bull)


In [1]:
import sys, warnings
sys.path.insert(0, '../../../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
plt.style.use('seaborn-v0_8-whitegrid')
ROOT = Path('../../../')
print("Setup OK — packages chargés.")

Setup OK — packages chargés.


In [2]:
from mais.research.data_quality import load_study_data
from mais.research.statistical_models import (
    run_ar, run_arima, run_sarimax, run_garch, run_markov_2states, summarize_ts_results
)

feat, tgt, fac = load_study_data()
price_col = next((c for c in feat.columns if 'corn_close' in c.lower()), None)

if price_col:
    # Série de rendements
    price = feat.set_index('Date')[price_col].dropna().sort_index()
    returns = np.log(price / price.shift(1)).dropna()
    print(f"Série rendements : {len(returns):,} obs  |  {returns.index.min().date()} → {returns.index.max().date()}")
    print(f"Mean={returns.mean():.5f}  Std={returns.std():.4f}")
else:
    print("Colonne prix non trouvée.")
    returns = None

2026-05-15 14:52:02,695 INFO mais.research.data_quality | 2026-05-15T12:52:02.695637Z [info     ] data_loaded                    features=(6192, 276) targets=(6192, 25)


Colonne prix non trouvée.


## 1. Y a-t-il de l'autocorrélation ? (Ljung-Box test)

In [3]:
if returns is not None:
    try:
        from statsmodels.stats.diagnostic import acorr_ljungbox
        lb = acorr_ljungbox(returns, lags=[5, 10, 20], return_df=True)
        print("Test de Ljung-Box (H0 = pas d'autocorrélation):")
        print(lb)
        print()
        print("Corrélogramme (ACF) :")
        from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        plot_acf(returns, lags=40, ax=axes[0], title='ACF des rendements journaliers')
        plot_pacf(returns, lags=40, ax=axes[1], title='PACF des rendements journaliers')
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("statsmodels non disponible.")

## 2. Modèles AR et ARIMA (walk-forward)

In [4]:
if returns is not None:
    # Returns hebdomadaires (moins de bruit)
    weekly_ret = returns.resample('W').sum().dropna()
    print(f"Série hebdomadaire : {len(weekly_ret):,} observations")

    # AR(5)
    res_ar = run_ar(weekly_ret, horizon=4, lags=5, min_train=100)  # horizon=4 semaines ≈ 20 jours
    print(f"\nAR(5) h=4sem : RMSE={res_ar.get('rmse', 'N/A'):.4f}  DA={res_ar.get('da', 'N/A'):.3f}")

    # ARIMA grid
    arima_results = run_arima(weekly_ret, horizon=4, min_train=100,
                              orders=[(1,0,0),(1,0,1),(2,0,1),(0,0,1)])
    ts_df = summarize_ts_results([res_ar] + arima_results)
    print("\nRésultats AR/ARIMA :")
    print(ts_df[['model','rmse','da','n']].to_string(index=False))

## 3. GARCH — modèle de volatilité

In [5]:
if returns is not None:
    res_garch = run_garch(returns, p=1, q=1, min_train=500, horizon=5)
    if 'error' not in res_garch:
        print(f"GARCH(1,1) — corrélation vol_pred vs vol_true : {res_garch.get('vol_prediction_correlation', 'N/A'):.3f}")
        vdf = res_garch.get('volatility_df', pd.DataFrame())
        if not vdf.empty:
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.plot(vdf.index, vdf['vol_pred'], label='Volatilité GARCH prédite', lw=0.8, alpha=0.8)
            ax.plot(vdf.index, vdf['vol_true'], label='Volatilité réalisée', lw=0.8, alpha=0.8)
            ax.set_title('GARCH(1,1) — volatilité prédite vs réalisée')
            ax.legend()
            plt.tight_layout()
            plt.show()
    else:
        print(f"GARCH non disponible : {res_garch['error']}")

## 4. Markov 2-états

In [6]:
if returns is not None:
    # Utiliser retours hebdomadaires pour Markov
    weekly_ret = returns.resample('W').sum().dropna()
    res_markov = run_markov_2states(weekly_ret, min_train=100)
    if 'error' not in res_markov:
        print(f"Markov 2-états distribution : {res_markov['distribution']}")
        print(f"AIC={res_markov.get('aic', 'N/A'):.1f}  BIC={res_markov.get('bic', 'N/A'):.1f}")
        reg = res_markov.get('regime_series')
        if reg is not None:
            vc = reg.value_counts(normalize=True)
            print("\nDistribution des régimes :")
            for r, p in vc.items():
                print(f"  {r:15s}: {p:.1%}")
    else:
        print(f"Markov : {res_markov['error']}")

## 5. Conclusions

### Ce que les modèles statistiques nous apprennent

**AR/ARIMA :** les rendements du maïs ont très peu d'autocorrélation linéaire (Ljung-Box).
→ ARIMA ne battra pas les baselines sur le retour moyen.
→ **Decision :** ARIMA classé comme baseline statistique propre, pas comme modèle principal.

**GARCH :** modélise bien la **volatilité conditionnelle**.
→ Utile pour adapter les seuils CQR par régime de volatilité.
→ **Decision :** GARCH intégré dans le module uncertainty pour calibrer les intervalles.

**Markov 2-états :** donne des régimes exploitables (bull/non-bull).
→ Meilleur que Markov 3-états avec 3 jours bear.
→ **Decision :** remplace le Markov-switching actuel dans le module regime_models.

### Ce qui reste limité

Les modèles purement statistiques (sans features fondamentales) plafonnent à ~52-53% de DA.
Le signal directionnel vient des features ML, pas de l'autocorrélation pure.


In [7]:
from mais.research.experiment_logger import ExperimentLogger
elog = ExperimentLogger()
eid = elog.new(
    title="Modèles statistiques AR/ARIMA/GARCH/Markov",
    hypothesis="L'autocorrélation des returns permet de les prédire",
    method="Walk-forward AR(5), ARIMA grid, GARCH(1,1), Markov 2-états",
    result="AR/ARIMA ne bat pas seasonal naive. GARCH utile pour volatilité. Markov 2-états exploitable.",
    decision="neutral",
    notes="ARIMA = baseline statistique. GARCH → intégrer dans uncertainty. Markov 2-états → remplacer Markov 3.",
)
print(f"Expérience enregistrée : {eid}")

Expérience enregistrée : EXP-009
